# XRD Strain Analysis Pipeline

One notebook controls the full analysis. The notebook reads the YAML config, loops over scans, and calls the existing `nxprocess_lab` / `nxrefine_lab` functions. Helper modules keep the notebook readable; they do not replace the lab code.

## Stage 1: Preparation

In [ ]:
from pathlib import Path
import sys
import numpy as np

# Assumption: launch this notebook from the XRD_Strain_Analysis_Pipeline directory.
repo_root = Path.cwd().resolve()
print(f"Project root: {repo_root}")
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from xrd_strain_pipeline.config import all_scan_settings, load_experiment_config, merged_scan_settings
from xrd_strain_pipeline.helper import background_per_voxel, cdw_series_inputs_from_saved_windows, integrate_momentum_roi, load_cdw_window_definition, momentum_integration_bounds_from_background, momentum_line_scan_background_levels, prepare_reference_momentum_binning, process_cdw_strain_series, select_or_load_cdw_window, select_transform_bin_background_patch, plot_cdw_series_detector_windows, plot_cdw_series_integration_diagnostics, load_cdw_series_summary, plot_final_cdw_voltage_overlay, plot_final_cdw_voltage_series
from xrd_strain_pipeline.lab_wrappers import add_lab_module_path, check_manual_peak_orientation, check_orientation, combine_local_transform, copy_reference_ub_to_strain_scans, create_scan_files, momentum_line_scans, generate_cdw_satellites, get_hkl_at_pixel, labcreate_kwargs, labcreate_output_paths, labreduce_kwargs, list_manual_peak_hkls, predict_bragg_positions, prepare_manual_ub, prepare_orientation, prepare_transform, q_vectors_from_config, select_manual_ub, select_orientation_grain, transform_local_window, ubcopy_kwargs, unitcell_kwargs, working_directory, write_manual_peaklists
from xrd_strain_pipeline.nexus_checks import validate_created, validate_postpeaks, validate_refined
from xrd_strain_pipeline.plotting import plot_bragg_predictions, plot_cdw_predictions, plot_detector_window, plot_lab_momentum_maps, plot_momentum_line_scans, plot_peaklist, resolve_overlay_png, select_detector_window_interactive
from xrd_strain_pipeline.window_utils import normalize_detector_window, save_window_definition

In [ ]:
#This loads the yaml config file and adds the lab module path to sys.path so that we can import the lab modules.
config_path = repo_root / "configs" / "experiment_template.yaml"
config = load_experiment_config(config_path)

lab_path = add_lab_module_path(repo_root, config["project"]["lab_module_path"])
print(f"Lab module path: {lab_path}")

import nxprocess_lab
import nxrefine_lab
from nexusformat.nexus import nxload

### Preparation Controls

Load and validate configuration, select scans, create NeXus/HDF5 files, and apply the refined unit cell.

In [ ]:
RUN_CREATE = True
RUN_VALIDATE_CREATED = True
RUN_UNITCELL_UPDATE = False

# While developing, use a subset such as ["scan_000_reference"].
# Use None to process every scan listed in the YAML.
SELECTED_SCAN_IDS = None

### Scan Selection

Every row comes from the YAML. The reference scan establishes UB; strain scans reuse it.

In [ ]:
all_settings = all_scan_settings(config)
if SELECTED_SCAN_IDS is not None:
    selected = set(SELECTED_SCAN_IDS)
    scan_settings = [scan for scan in all_settings if scan["scan_id"] in selected]
else:
    scan_settings = all_settings

reference_settings = merged_scan_settings(config, config["reference_scan_id"])
scan_groups = {scan["scan_id"]: [scan] for scan in all_settings}

print(f"Reference scan: {reference_settings['scan_id']} -> {reference_settings['file']}")
print(f"Scans selected: {len(scan_settings)}")
for scan in scan_settings:
    print(
        f"{scan['scan_id']:>20} | role={scan['role']:<9} | "
        f"strain={scan['strain']} | file={scan['file']}"
    )


### Argument Preview

This cell builds the arguments for each lab function without writing data. If this fails, the YAML is missing something important.

In [ ]:
prepared = {}
for scan in scan_settings:
    scan_id = scan["scan_id"]
    prepared[scan_id] = {
        "settings": scan,
        "create": labcreate_kwargs(scan),
        "unitcell": unitcell_kwargs(scan),
        "reduce": labreduce_kwargs(scan, threshold=None),
    }
    if scan["role"] != "reference":
        prepared[scan_id]["ubcopy"] = ubcopy_kwargs(reference_settings, scan, repo_root)

for scan_id, stage_args in prepared.items():
    create = stage_args["create"]
    print(f"{scan_id}: file={create['file']}, TIFF base={create['data_base_name']}, images={create['imstart']}-{create['imend']}, output={stage_args['settings']['output_directory']}")

### Create `.nxs` / `.hdf5`

Calls `nxprocess_lab.LabCreate` for each selected scan.

In [ ]:

if RUN_CREATE:
    create_scan_files(prepared, repo_root, nxprocess_lab)
else:
    print("Dry run only. Set RUN_CREATE = True to create .nxs/.hdf5 files.")

### Validate Created Files

In [ ]:
if RUN_VALIDATE_CREATED:
    for scan_id, stage_args in prepared.items():
        settings = stage_args["settings"]
        file = repo_root / settings["output_directory"] / (settings["file"] + ".nxs")
        if not file.exists():
            print(f"{scan_id}: missing .nxs, skipped validation. Run RUN_CREATE = True first: {file}")
            continue
        nxroot = nxload(str(file), "r")
        result = validate_created(nxroot)
        print(scan_id, result)
else:
    print("Skipped created-file validation.")

### Apply Refined Unit Cell

Writes the YAML unit-cell values into `/entry/sample/unitcell_*` for each selected scan.

In [ ]:
if RUN_UNITCELL_UPDATE:
    for scan_id, stage_args in prepared.items():
        output_dir = repo_root / stage_args["settings"]["output_directory"]
        print(f"Updating unit cell for {scan_id}")
        with working_directory(output_dir):
            nxprocess_lab.LabReduce_unitcell(**stage_args["unitcell"])
else:
    print("Dry run only. Set RUN_UNITCELL_UPDATE = True to write unit-cell values.")

## Stage 2: Peak Finding and Peak Lists

Use optional automatic peak finding or load curated manual peak lists into `/entry/postpeaks`.

In [ ]:
RUN_REDUCE = False
RUN_MANUAL_PEAKLIST = False

### Optional Automatic Peak Finding

Calls `nxprocess_lab.LabReduce`. Keep this off when Be dome artifacts make automatic detection unreliable.

In [ ]:
peaks_by_scan = {}

if RUN_REDUCE:
    for scan_id, stage_args in prepared.items():
        print(f"Reducing/finding peaks for {scan_id}")
        output_dir = repo_root / stage_args["settings"]["output_directory"]
        with working_directory(output_dir):
            peaks_by_scan[scan_id] = nxprocess_lab.LabReduce(
                **stage_args["reduce"],
                peaklist_filename=f"{scan_id}_peaklist.txt",
            )
else:
    print("Dry run only. Set RUN_REDUCE = True to run auto peak finding.")

### Inspect Manual Peak Lists

Plot a curated peak-list file before writing it into `/entry/postpeaks`. Set `overlay_png` to a detector PNG if you want image context behind the selected peaks.

In [ ]:
peaklist_path = repo_root / "Peak_Lists" / "scan_000_reference_peaks.txt"
overlay_png = repo_root / config["project"]["overlay_root"] / "scan_000_reference.png"
graphs_dir = repo_root / config["project"]["output_root"] / "Graphs"
save_path = graphs_dir / f"{peaklist_path.stem}_peaklist_check.png"
# Set overlay_png = None if you only want the scatter plot.

fig, ax, displayed_peaklist = plot_peaklist(peaklist_path, overlay_png=overlay_png, save_path=save_path)
print(f"Saved graph: {save_path}")

### Load Manual Peak Lists

Preferred when auto peak finding is unreliable. Each `manual_peaklist_file` should be a TXT or CSV file with columns `[z_frame, y, x, intensity]`. This writes `/entry/postpeaks` using `nxprocess_lab.LabReduce_peaklist`.

In [ ]:
if RUN_MANUAL_PEAKLIST:
    write_manual_peaklists(prepared, repo_root, nxprocess_lab, np)
else:
    print("Dry run only. Set RUN_MANUAL_PEAKLIST = True to write curated peak lists.")

## Stage 3: UB Matrix Determination

Try automatic orientation first. If it fails or projects poorly, use the manual two-peak UB workflow below.

### Automatic UB Search

In [ ]:
RUN_ORIENTATION_PREPARE = False
RUN_ORIENTATION_CHECK = False
RUN_ORIENTATION_SELECT = False
SELECTED_GRAIN = 0

In [ ]:
refinevars_by_scan = {}

if RUN_ORIENTATION_PREPARE:
    refinevars_by_scan[reference_settings["scan_id"]] = prepare_orientation(
        reference_settings,
        repo_root,
        nxprocess_lab,
        hklselect=True,
        peak_tolerance=5.0,
        ring_tolerance=1.0,
        hkl_tolerance=0.05,
        mode=1,
    )
else:
    print("Dry run only. Set RUN_ORIENTATION_PREPARE = True to find orientation candidates.")

In [ ]:

if RUN_ORIENTATION_CHECK:
    refinevars = refinevars_by_scan[reference_settings["scan_id"]]
    check_orientation(
        refinevars,
        nxprocess_lab,
        hkl_tolerance=0.1,
        write_peaklist=True,
    )
else:
    print("Dry run only. Set RUN_ORIENTATION_CHECK = True to inspect orientation candidates.")

In [ ]:
if RUN_ORIENTATION_SELECT:
    refinevars = refinevars_by_scan[reference_settings["scan_id"]]
    select_orientation_grain(refinevars, nxprocess_lab, grain=SELECTED_GRAIN)
else:
    print("Dry run only. Set RUN_ORIENTATION_SELECT = True after choosing SELECTED_GRAIN.")

### Manual UB Fallback

Use this when the automatic orientation candidates are wrong. The helper below loads the reference NeXus file and `/entry/postpeaks` if `refinevars_by_scan` does not already contain the reference scan.

In [ ]:
RUN_MANUAL_HKL_LIST = True
RUN_MANUAL_ORIENT_CHECK = True
RUN_MANUAL_UB_PREPARE = True
RUN_MANUAL_UB_SELECT = True

MANUAL_PEAK_I = 0
MANUAL_PEAK_J = 1
MANUAL_RING_TOLERANCE = 0
MANUAL_SELECTED_UB = 0

In [ ]:
def _reference_refinevars():
    global refinevars_by_scan
    if "refinevars_by_scan" not in globals():
        refinevars_by_scan = {}
    scan_id = reference_settings["scan_id"]
    if scan_id not in refinevars_by_scan:
        print(f"Preparing refinevars for {scan_id} using manual orientation mode")
        refinevars_by_scan[scan_id] = prepare_orientation(
            reference_settings,
            repo_root,
            nxprocess_lab,
            hklselect=False,
            peak_tolerance=5.0,
            ring_tolerance=1.0,
            hkl_tolerance=0.05,
            mode=1,
        )
    return refinevars_by_scan[scan_id]

In [ ]:
if RUN_MANUAL_HKL_LIST:
    refinevars = _reference_refinevars()
    ring_tolerance = 0 if MANUAL_RING_TOLERANCE is None else MANUAL_RING_TOLERANCE
    list_manual_peak_hkls(
        refinevars,
        nxprocess_lab,
        i=MANUAL_PEAK_I,
        j=MANUAL_PEAK_J,
        ring_tolerance=ring_tolerance,
    )
else:
    print("Skipped manual HKL listing. Set RUN_MANUAL_HKL_LIST = True after choosing two peak indices.")

In [ ]:
if RUN_MANUAL_ORIENT_CHECK:
    refinevars = _reference_refinevars()
    check_manual_peak_orientation(
        refinevars,
        nxprocess_lab,
        i=MANUAL_PEAK_I,
        j=MANUAL_PEAK_J,
        i_ring=0,
        j_ring=0,
    )
else:
    print("Skipped manual orientation check. Set RUN_MANUAL_ORIENT_CHECK = True after choosing ring assignments.")

In [ ]:
manual_ulists_by_scan = {}

if RUN_MANUAL_UB_PREPARE:
    scan_id = reference_settings["scan_id"]
    refinevars = _reference_refinevars()
    manual_ulists_by_scan[scan_id] = prepare_manual_ub(
        refinevars,
        nxprocess_lab,
        i=MANUAL_PEAK_I,
        j=MANUAL_PEAK_J,
        hkl_tolerance=0.1,
        write_peaklist=True,
        ring_tolerance=MANUAL_RING_TOLERANCE,
    )
else:
    print("Skipped manual UB preparation. Set RUN_MANUAL_UB_PREPARE = True after the two peaks look right.")

In [ ]:
if RUN_MANUAL_UB_SELECT:
    scan_id = reference_settings["scan_id"]
    refinevars = _reference_refinevars()
    if scan_id not in manual_ulists_by_scan:
        raise RuntimeError("Run the manual UB preparation cell first so manual_ulists_by_scan has candidates.")
    ulist = manual_ulists_by_scan[scan_id]
    select_manual_ub(refinevars, nxprocess_lab, ulist, iU=MANUAL_SELECTED_UB)
else:
    print("Skipped manual UB selection. Set RUN_MANUAL_UB_SELECT = True after choosing MANUAL_SELECTED_UB.")

## Stage 4: Projection and UB Validation

Project Bragg and CDW targets through the selected UB matrix, then copy the accepted UB and validate refined state.

In [ ]:
RUN_BRAGG_PROJECTION = False
RUN_CDW_PROJECTION = False
# Set True after the reference UB/orientation matrix is finalized.
# This copies the reference UB to every non-reference scan in the current YAML selection.
RUN_COPY_REFERENCE_UB = False
RUN_VALIDATE_REFINED = False


# Projection-only frame acceptance. Use 500 for 25 degrees with 0.05 deg/frame.
CDW_PROJECTION_FRAME_MIN = None
CDW_PROJECTION_FRAME_MAX = 500


### Bragg Projection Check

In [ ]:
get_hkl_at_pixel(reference_settings, repo_root, nxprocess_lab, x = 2396, y = 1601, z_frame = 171)

In [ ]:
file_no_ext = str(file)
if file_no_ext.endswith(".nxs"):
    file_no_ext = file_no_ext[:-4]

for h, k, l in [(1, -3/2, -9/2), (1/2, -1, -7/2), (-1/2, 1, -7/2)]:
    peaks = nxprocess_lab.LabRefine_getxyz(file=file_no_ext, h=h, k=k, l=l)
    print(f"HKL=({h}, {k}, {l})")
    for peak in peaks:
        print(f"  x={peak.x:.2f}, y={peak.y:.2f}, z_index={peak.z:.2f}")

In [ ]:
bragg_predictions = []
if RUN_BRAGG_PROJECTION:
    bragg_list = []
    for index, hkl in enumerate(config["analysis"]["bragg_projection_list"]):
        if hkl is None or len(hkl) != 3:
            raise ValueError(f"Bad bragg_projection_list entry at index {index}: {hkl!r}")
        bragg_list.append(tuple(hkl))
    bragg_predictions = predict_bragg_positions(reference_settings, repo_root, nxprocess_lab, bragg_list)
    overlay_png = repo_root / config["project"]["overlay_root"] / "scan_000_reference.png"
    graphs_dir = repo_root / config["project"]["output_root"] / "Graphs"
    save_path = graphs_dir / f"{reference_settings['scan_id']}_bragg_projection_check.png"
    fig, ax = plot_bragg_predictions(bragg_predictions, overlay_png=overlay_png, save_path=save_path)
    print(f"Projected {len(bragg_predictions)} Bragg positions")
else:
    print("Dry run only. Set RUN_BRAGG_PROJECTION = True after selecting an orientation grain.")

### CDW Satellite Projection

Generate CDW satellite targets from the configured Bragg HKLs and YAML q vectors, then project them onto the detector overlay using the current reference UB matrix.

In [ ]:
cdw_predictions = []
cdw_satellites = []

if RUN_CDW_PROJECTION:
    bragg_list = []
    for index, hkl in enumerate(config["analysis"]["bragg_projection_list"]):
        if hkl is None or len(hkl) != 3:
            raise ValueError(f"Bad bragg_projection_list entry at index {index}: {hkl!r}")
        bragg_list.append(tuple(hkl))

    q_vectors = q_vectors_from_config(config)
    if not q_vectors:
        raise ValueError("No q_vectors configured in the YAML")
    if all(np.allclose(q, (0.0, 0.0, 0.0)) for q in q_vectors):
        print("Warning: all q_vectors are zero; update q_vectors in the YAML for real CDW prediction.")

    cdw_satellites = generate_cdw_satellites(bragg_list, q_vectors)
    bragg_predictions = predict_bragg_positions(
        reference_settings,
        repo_root,
        nxprocess_lab,
        bragg_list,
        frame_min=CDW_PROJECTION_FRAME_MIN,
        frame_max=CDW_PROJECTION_FRAME_MAX,
    )
    cdw_predictions = predict_bragg_positions(
        reference_settings,
        repo_root,
        nxprocess_lab,
        cdw_satellites,
        frame_min=CDW_PROJECTION_FRAME_MIN,
        frame_max=CDW_PROJECTION_FRAME_MAX,
    )

    overlay_png = repo_root / config["project"]["overlay_root"] / "demo_strain000.png"
    graphs_dir = repo_root / config["project"]["output_root"] / "Graphs"
    save_path = graphs_dir / f"{reference_settings['scan_id']}_cdw_projection_check_000.png"
    fig, ax = plot_cdw_predictions(
        bragg_predictions,
        cdw_predictions,
        overlay_png=overlay_png,
        save_path=save_path,
        show_bragg_labels=True,
        show_cdw_labels=False,
    )
    print(f"Configured {len(q_vectors)} q vectors")
    print(f"Generated {len(cdw_satellites)} CDW satellite HKLs")
    print(f"Projection frame acceptance: {CDW_PROJECTION_FRAME_MIN} to {CDW_PROJECTION_FRAME_MAX}")
    print(f"Projected {len(cdw_predictions)} CDW satellite positions")
    print(f"Saved graph: {save_path}")
else:
    print("Dry run only. Set RUN_CDW_PROJECTION = True after the reference UB is finalized.")

### Copy Reference UB To Strain Scans

After the reference orientation matrix is finalized, copy it to all strain scans.

In [ ]:
if RUN_COPY_REFERENCE_UB:
    ub_copy_summary = copy_reference_ub_to_strain_scans(
        prepared,
        repo_root=repo_root,
        nxprocess_lab=nxprocess_lab,
        reference_scan_id=config["reference_scan_id"],
    )
    print(
        "UB copy summary: "
        f"copied={len(ub_copy_summary['copied'])}, "
        f"missing={len(ub_copy_summary['missing'])}, "
        f"reference_skipped={len(ub_copy_summary['skipped_reference'])}"
    )
else:
    print("Skipped UB copy. Set RUN_COPY_REFERENCE_UB = True after the reference UB is finalized.")

### Validate Refined/HKL State

Use after UB has been selected/copied and HKL values have been written.

In [ ]:
if RUN_VALIDATE_REFINED:
    for scan_id, stage_args in prepared.items():
        settings = stage_args["settings"]
        file = repo_root / settings["output_directory"] / (settings["file"] + ".nxs")
        if not file.exists():
            print(f"{scan_id}: missing .nxs, skipped refined-state validation: {file}")
            continue
        nxroot = nxload(str(file), "r")
        result = validate_refined(nxroot)
        print(scan_id, result)
else:
    print("Skipped refined-state validation.")

## Stage 5: Reference Reciprocal-Space Transformation

Prepare transform metadata first. Local CDW detector windows and H/K/L integration will follow in this stage.

In [ ]:
RUN_TRANSFORM_PREPARE = False

RUN_INTERACTIVE_WINDOW_SELECTOR = False
RUN_LOCAL_TRANSFORM = True

### Prepare Transform Metadata

Set `/entry/data:last`, estimate detector-voxel changes in H/K/L, and write `/entry/transform_prepare`. This does not yet transform intensity data.

In [ ]:
transform_config = config["analysis"].get("transform_preparation", {})

if RUN_TRANSFORM_PREPARE:
    transform_prepare_results = {}
    for scan_id, stage_args in prepared.items():
        print(f"Preparing reciprocal-space transform metadata for {scan_id}")
        result = prepare_transform(
            stage_args["settings"],
            repo_root,
            nxprocess_lab,
            itn=transform_config.get("iterations", 1000),
            chunkx=transform_config.get("chunkx", 5),
            chunky=transform_config.get("chunky", 5),
            chunkz=transform_config.get("chunkz", 1),
        )
        transform_prepare_results[scan_id] = result
        print(result)
else:
    print("Skipped transform preparation. Review YAML settings, then set RUN_TRANSFORM_PREPARE = True.")

### Select and Preview a Local Detector Window

Define a small Albula-coordinate window around one peak. Previewing the rectangle is independent of running the local transform. X increases right, Y increases downward, and Z bounds are real frame numbers.

In [ ]:
# Pick the CDW family you want to inspect or process.
# If configs/peak_windows/<CDW>_signal.json exists, the saved bounds are loaded automatically.
WINDOW_CDW_ID = "CDW1"

RUN_INTERACTIVE_WINDOW_SELECTOR = False  # True only when intentionally selecting/reselecting the signal window.
USE_SAVED_CDW_WINDOW = True
RUN_LOCAL_TRANSFORM = False  # Demo default: turn on only after private data are available.

CDW_REFERENCE_SCAN_IDS = {
    "CDW1": "scan_001_strain000_CDW1",
}
CDW_OVERLAY_FILES = {
    "CDW1": "demo_strain000.png",
}

# Fallback demo guesses are used only when no saved window exists yet.
# Replace these with detector coordinates for your private data.
CDW_INITIAL_GUESSES = {
    "CDW1": {
        "center": (1700, 1700, 200),
        "signal_bounds": (1650, 1750, 1650, 1750, 190, 210),
        "background_bounds": (1500, 1600, 1500, 1600, 190, 210),
        "deltas": (0.0025, 0.0025, 0.0030),
    },
}

LOCAL_TRANSFORM_SCAN_ID = CDW_REFERENCE_SCAN_IDS[WINDOW_CDW_ID]
LOCAL_PEAK_ID = WINDOW_CDW_ID
LOCAL_TARGET_HKL = None
LOCAL_PARENT_BRAGG_HKL = None
LOCAL_OVERLAY_FILE = CDW_OVERLAY_FILES.get(WINDOW_CDW_ID)

_initial = CDW_INITIAL_GUESSES[WINDOW_CDW_ID]
PEAK_X, PEAK_Y, PEAK_Z_FRAME = _initial["center"]
XSTART, XEND, YSTART, YEND, ZSTART, ZEND = _initial["signal_bounds"]
MOMENTUM_DELTAH, MOMENTUM_DELTAK, MOMENTUM_DELTAL = _initial["deltas"]


In [ ]:
local_settings = prepared[LOCAL_TRANSFORM_SCAN_ID]["settings"]

signal_window = select_or_load_cdw_window(
    settings=local_settings,
    repo_root=repo_root,
    config=config,
    cdw_id=WINDOW_CDW_ID,
    window_role="signal",
    initial_center=(PEAK_X, PEAK_Y, PEAK_Z_FRAME),
    initial_bounds=(XSTART, XEND, YSTART, YEND, ZSTART, ZEND),
    overlay_file=LOCAL_OVERLAY_FILE,
    use_saved=USE_SAVED_CDW_WINDOW,
    run_interactive_selector=RUN_INTERACTIVE_WINDOW_SELECTOR,
    target_hkl=LOCAL_TARGET_HKL,
    parent_bragg_hkl=LOCAL_PARENT_BRAGG_HKL,
)

PEAK_X, PEAK_Y, PEAK_Z_FRAME = signal_window["center"]
XSTART, XEND, YSTART, YEND, ZSTART, ZEND = signal_window["bounds"]

print(f"Loaded {WINDOW_CDW_ID} signal window from {signal_window['source']} values")
print(f"Pixel window: x=[{XSTART}, {XEND}), y=[{YSTART}, {YEND}), z frames=[{ZSTART}, {ZEND})")
print(f"Window size: {XEND-XSTART} x {YEND-YSTART} x {ZEND-ZSTART} voxels")
print(f"Saved preview: {signal_window['preview_path']}")
print(f"Saved reusable window definition: {signal_window['path']}")

### Run Local Pixel-to-HKL Transform

In [ ]:
if RUN_LOCAL_TRANSFORM:
    local_settings = prepared[LOCAL_TRANSFORM_SCAN_ID]["settings"]
    local_transform_result = transform_local_window(
        local_settings,
        repo_root,
        nxprocess_lab,
        peak_id=LOCAL_PEAK_ID,
        xstart=XSTART, xend=XEND,
        ystart=YSTART, yend=YEND,
        zstart=ZSTART, zend=ZEND,
        center=(PEAK_X, PEAK_Y, PEAK_Z_FRAME),
    )
    data_t = local_transform_result["data_t"]
    _h = local_transform_result["h"]
    _k = local_transform_result["k"]
    _l = local_transform_result["l"]
    print(f"Saved local transform: {local_transform_result['save_path']}")
else:
    print("Preview only. Set RUN_LOCAL_TRANSFORM = True after accepting the detector window.")

## Stage 6: Reference Momentum-Space Analysis

Use each reference `<peak_id>_transform_local.npz` to choose H/K/L bin sizes, inspect momentum maps/scans, and save objective ROI/background parameters for that peak.

### Bin Reference Transform

Estimate reciprocal-space bin sizes from the saved local H/K/L transform, then re-bin intensity onto a regular momentum grid.

In [ ]:
RUN_REFERENCE_MOMENTUM_BIN_ESTIMATE = False
RUN_REFERENCE_MOMENTUM_BINNING = True
RUN_REFERENCE_MOMENTUM_MAPS = True

REFERENCE_MOMENTUM_SCAN_ID = LOCAL_TRANSFORM_SCAN_ID
REFERENCE_MOMENTUM_PEAK_ID = LOCAL_PEAK_ID

# Use None while estimating, then set fixed values and turn RUN_REFERENCE_MOMENTUM_BIN_ESTIMATE off.
MOMENTUM_DELTAH = 0.0026 # Example: 0.002
MOMENTUM_DELTAK = 0.0022  # Example: 0.0025
MOMENTUM_DELTAL = 0.0033  # Example: 0.0055

# Plane values default to the brightest voxel in the binned grid.
MOMENTUM_HVAL = None
MOMENTUM_KVAL = None
MOMENTUM_LVAL = None

RUN_REFERENCE_MOMENTUM_SCANS = True

# Line-scan ranges default to the full binned range. Set values to zoom near the peak.
MOMENTUM_HSCAN_MIN = None
MOMENTUM_HSCAN_MAX = None
MOMENTUM_KSCAN_MIN = None
MOMENTUM_KSCAN_MAX = None
MOMENTUM_LSCAN_MIN = None
MOMENTUM_LSCAN_MAX = None

# Half-widths in binned voxels summed perpendicular to the line-scan axis.
MOMENTUM_SCAN_HSTEP = 2
MOMENTUM_SCAN_KSTEP = 2
MOMENTUM_SCAN_LSTEP = 2

RUN_REFERENCE_MOMENTUM_INTEGRATION = True

# These default to the background-intersection bounds from the line-scan cell.
# Set fixed values here if you want to override them manually.
MOMENTUM_INTEGRATION_HMIN = None
MOMENTUM_INTEGRATION_HMAX = None
MOMENTUM_INTEGRATION_KMIN = None
MOMENTUM_INTEGRATION_KMAX = None
MOMENTUM_INTEGRATION_LMIN = None
MOMENTUM_INTEGRATION_LMAX = None


In [ ]:
reference_momentum_prep = prepare_reference_momentum_binning(
    prepared=prepared,
    repo_root=repo_root,
    scan_id=REFERENCE_MOMENTUM_SCAN_ID,
    peak_id=REFERENCE_MOMENTUM_PEAK_ID,
    deltah=MOMENTUM_DELTAH,
    deltak=MOMENTUM_DELTAK,
    deltal=MOMENTUM_DELTAL,
    run_estimate=RUN_REFERENCE_MOMENTUM_BIN_ESTIMATE,
)

reference_momentum_settings = reference_momentum_prep["settings"]
reference_transform_path = reference_momentum_prep["local_transform_path"]
MOMENTUM_DELTAH = reference_momentum_prep["deltah"]
MOMENTUM_DELTAK = reference_momentum_prep["deltak"]
MOMENTUM_DELTAL = reference_momentum_prep["deltal"]
bin_estimate = reference_momentum_prep["bin_estimate"]


In [ ]:
if RUN_REFERENCE_MOMENTUM_BINNING:
    reference_momentum_result = combine_local_transform(
        reference_momentum_settings,
        repo_root,
        nxprocess_lab,
        peak_id=REFERENCE_MOMENTUM_PEAK_ID,
        deltah=MOMENTUM_DELTAH,
        deltak=MOMENTUM_DELTAK,
        deltal=MOMENTUM_DELTAL,
        local_transform_path=reference_transform_path,
    )
    transform_data = reference_momentum_result["transform_data"]
    Hrange = reference_momentum_result["Hrange"]
    Krange = reference_momentum_result["Krange"]
    Lrange = reference_momentum_result["Lrange"]
    print(f"Saved binned momentum grid: {reference_momentum_result['save_path']}")
    print(f"Grid shape: {transform_data.shape}; brightest voxel HKL={reference_momentum_result['max_hkl']}")
else:
    print("Skipped binning. Set RUN_REFERENCE_MOMENTUM_BINNING = True to create the momentum grid.")

In [ ]:
if RUN_REFERENCE_MOMENTUM_MAPS:
    reference_maps_dir = (
        repo_root
        / config["project"]["output_root"]
        / "Graphs"
        / REFERENCE_MOMENTUM_SCAN_ID
        / REFERENCE_MOMENTUM_PEAK_ID
    )
    reference_map_result = plot_lab_momentum_maps(
        nxprocess_lab,
        transform_data=transform_data,
        hrange=Hrange,
        krange=Krange,
        lrange=Lrange,
        hval=MOMENTUM_HVAL,
        kval=MOMENTUM_KVAL,
        lval=MOMENTUM_LVAL,
        save_dir=reference_maps_dir,
        filename_prefix=f"{REFERENCE_MOMENTUM_PEAK_ID}_momentum",
    )
    print(f"Saved momentum maps under: {reference_maps_dir}")
else:
    print("Skipped maps. Set RUN_REFERENCE_MOMENTUM_MAPS = True after binning.")

### Background Detector Patch

Choose a second detector/frame patch away from the CDW feature. This patch is transformed and binned with the same momentum bin sizes, then summarized as background counts per occupied momentum voxel.

In [ ]:
RUN_BACKGROUND_PATCH_TRANSFORM = True
RUN_BACKGROUND_PATCH_INTERACTIVE_SELECTOR = False  # True only when intentionally reselecting the background window.
USE_SAVED_BACKGROUND_WINDOW = True

BACKGROUND_PATCH_ID = f"{WINDOW_CDW_ID}_background"
BACKGROUND_OVERLAY_FILE = LOCAL_OVERLAY_FILE

_initial = CDW_INITIAL_GUESSES[WINDOW_CDW_ID]
BACKGROUND_PEAK_X, BACKGROUND_PEAK_Y, BACKGROUND_PEAK_Z_FRAME = _initial["center"]
(
    BACKGROUND_XSTART,
    BACKGROUND_XEND,
    BACKGROUND_YSTART,
    BACKGROUND_YEND,
    BACKGROUND_ZSTART,
    BACKGROUND_ZEND,
) = _initial["background_bounds"]

In [ ]:
if RUN_BACKGROUND_PATCH_TRANSFORM:
    background_window = select_or_load_cdw_window(
        settings=reference_momentum_settings,
        repo_root=repo_root,
        config=config,
        cdw_id=WINDOW_CDW_ID,
        window_role="background",
        initial_center=(BACKGROUND_PEAK_X, BACKGROUND_PEAK_Y, BACKGROUND_PEAK_Z_FRAME),
        initial_bounds=(
            BACKGROUND_XSTART,
            BACKGROUND_XEND,
            BACKGROUND_YSTART,
            BACKGROUND_YEND,
            BACKGROUND_ZSTART,
            BACKGROUND_ZEND,
            
        ),
        overlay_file=BACKGROUND_OVERLAY_FILE,
        use_saved=USE_SAVED_BACKGROUND_WINDOW,
        run_interactive_selector=RUN_BACKGROUND_PATCH_INTERACTIVE_SELECTOR,
        require_saved_or_interactive=True,
    )

    BACKGROUND_PEAK_X, BACKGROUND_PEAK_Y, BACKGROUND_PEAK_Z_FRAME = background_window["center"]
    (
        BACKGROUND_XSTART,
        BACKGROUND_XEND,
        BACKGROUND_YSTART,
        BACKGROUND_YEND,
        BACKGROUND_ZSTART,
        BACKGROUND_ZEND,
    ) = background_window["bounds"]

    transforms_dir = repo_root / reference_momentum_settings["output_directory"] / "Transforms"
    background_local_path = transforms_dir / f"{BACKGROUND_PATCH_ID}_transform_local.npz"
    background_grid_path = transforms_dir / f"{BACKGROUND_PATCH_ID}_background_momentum_grid.npz"
    background_summary_path = transforms_dir / f"{BACKGROUND_PATCH_ID}_background_summary.json"

    background_local_transform = transform_local_window(
        reference_momentum_settings,
        repo_root,
        nxprocess_lab,
        peak_id=BACKGROUND_PATCH_ID,
        xstart=BACKGROUND_XSTART,
        xend=BACKGROUND_XEND,
        ystart=BACKGROUND_YSTART,
        yend=BACKGROUND_YEND,
        zstart=BACKGROUND_ZSTART,
        zend=BACKGROUND_ZEND,
        center=(BACKGROUND_PEAK_X, BACKGROUND_PEAK_Y, BACKGROUND_PEAK_Z_FRAME),
        save_path=background_local_path,
    )
    background_momentum_grid = combine_local_transform(
        reference_momentum_settings,
        repo_root,
        nxprocess_lab,
        peak_id=BACKGROUND_PATCH_ID,
        deltah=MOMENTUM_DELTAH,
        deltak=MOMENTUM_DELTAK,
        deltal=MOMENTUM_DELTAL,
        local_transform_path=background_local_transform["save_path"],
        save_path=background_grid_path,
    )
    background_summary = background_per_voxel(
        background_local_transform["save_path"],
        deltah=MOMENTUM_DELTAH,
        deltak=MOMENTUM_DELTAK,
        deltal=MOMENTUM_DELTAL,
        save_path=background_summary_path,
    )

    background_patch_result = {
        "window_path": background_window["path"],
        "preview_path": background_window["preview_path"],
        "local_transform": background_local_transform,
        "momentum_grid": background_momentum_grid,
        "background": background_summary,
    }
    print(f"Loaded {WINDOW_CDW_ID} background window from {background_window['source']} values")
    print(f"Saved background detector window: {background_patch_result['window_path']}")
    print(f"Saved background preview: {background_patch_result['preview_path']}")
    print(f"Saved background transform: {background_patch_result['local_transform']['save_path']}")
    print(f"Saved background momentum grid: {background_patch_result['momentum_grid']['save_path']}")
    print(
        "Background per occupied momentum voxel: "
        f"mean={background_summary['mean_counts_per_occupied_momentum_voxel']:.6g}, "
        f"median={background_summary['median_counts_per_occupied_momentum_voxel']:.6g}, "
        f"std={background_summary['std_counts_per_occupied_momentum_voxel']:.6g}"
    )
    print(
        "Background per detector pixel: "
        f"mean={background_summary['mean_counts_per_detector_pixel']:.6g}, "
        f"median={background_summary['median_counts_per_detector_pixel']:.6g}"
    )
else:
    print("Skipped background detector patch transform. Set RUN_BACKGROUND_PATCH_TRANSFORM = True after choosing the patch bounds.")

In [ ]:
if RUN_REFERENCE_MOMENTUM_SCANS:
    reference_scan_path = (
        repo_root
        / reference_momentum_settings["output_directory"]
        / "Transforms"
        / f"{REFERENCE_MOMENTUM_PEAK_ID}_momentum_line_scans.npz"
    )
    reference_scan_fig_path = (
        repo_root
        / config["project"]["output_root"]
        / "Graphs"
        / REFERENCE_MOMENTUM_SCAN_ID
        / REFERENCE_MOMENTUM_PEAK_ID
        / f"{REFERENCE_MOMENTUM_PEAK_ID}_momentum_line_scans.png"
    )
    reference_line_scans = momentum_line_scans(
        transform_data=transform_data,
        Hrange=Hrange,
        Krange=Krange,
        Lrange=Lrange,
        hmin=MOMENTUM_HSCAN_MIN,
        hmax=MOMENTUM_HSCAN_MAX,
        kmin=MOMENTUM_KSCAN_MIN,
        kmax=MOMENTUM_KSCAN_MAX,
        lmin=MOMENTUM_LSCAN_MIN,
        lmax=MOMENTUM_LSCAN_MAX,
        hstep=MOMENTUM_SCAN_HSTEP,
        kstep=MOMENTUM_SCAN_KSTEP,
        lstep=MOMENTUM_SCAN_LSTEP,
        save_path=reference_scan_path,
    )
    background_voxel_level = None
    background_line_levels = None
    if "background_summary" in globals():
        background_voxel_level = background_summary["mean_counts_per_occupied_momentum_voxel"]
        background_line_levels = momentum_line_scan_background_levels(reference_line_scans, background_voxel_level)
        print(f"Background per occupied momentum voxel: {background_voxel_level:.6g}")

    fig, axes = plot_momentum_line_scans(
        reference_line_scans,
        save_path=reference_scan_fig_path,
        background_y=background_line_levels,
        background_label="Scaled background",
    )
    integration_defaults = momentum_integration_bounds_from_background(
        reference_line_scans,
        background_line_levels,
    )
    if integration_defaults is not None:
        bounds = integration_defaults["bounds"]
        MOMENTUM_INTEGRATION_HMIN = bounds["hmin"] if MOMENTUM_INTEGRATION_HMIN is None else MOMENTUM_INTEGRATION_HMIN
        MOMENTUM_INTEGRATION_HMAX = bounds["hmax"] if MOMENTUM_INTEGRATION_HMAX is None else MOMENTUM_INTEGRATION_HMAX
        MOMENTUM_INTEGRATION_KMIN = bounds["kmin"] if MOMENTUM_INTEGRATION_KMIN is None else MOMENTUM_INTEGRATION_KMIN
        MOMENTUM_INTEGRATION_KMAX = bounds["kmax"] if MOMENTUM_INTEGRATION_KMAX is None else MOMENTUM_INTEGRATION_KMAX
        MOMENTUM_INTEGRATION_LMIN = bounds["lmin"] if MOMENTUM_INTEGRATION_LMIN is None else MOMENTUM_INTEGRATION_LMIN
        MOMENTUM_INTEGRATION_LMAX = bounds["lmax"] if MOMENTUM_INTEGRATION_LMAX is None else MOMENTUM_INTEGRATION_LMAX
    print(f"Line-scan center HKL: {reference_line_scans['center_hkl']}")
    print(f"Saved momentum line scans: {reference_scan_path}")
    print(f"Saved line-scan figure: {reference_scan_fig_path}")
else:
    print("Skipped momentum line scans. Set RUN_REFERENCE_MOMENTUM_SCANS = True after binning.")


In [ ]:
# Manual HKL integration ROI override.
# Set RUN_MANUAL_MOMENTUM_INTEGRATION_BOUNDS = True to use these values.

RUN_MANUAL_MOMENTUM_INTEGRATION_BOUNDS = True


if not RUN_MANUAL_MOMENTUM_INTEGRATION_BOUNDS:
    MOMENTUM_INTEGRATION_HMIN = None
    MOMENTUM_INTEGRATION_HMAX = None
    MOMENTUM_INTEGRATION_KMIN = None
    MOMENTUM_INTEGRATION_KMAX = None
    MOMENTUM_INTEGRATION_LMIN = None
    MOMENTUM_INTEGRATION_LMAX = None

else:
    MOMENTUM_INTEGRATION_HMIN = 0.489216
    MOMENTUM_INTEGRATION_HMAX = 0.512139
    MOMENTUM_INTEGRATION_KMIN = -1.015959
    MOMENTUM_INTEGRATION_KMAX = -0.983512
    MOMENTUM_INTEGRATION_LMIN = -3.560591
    MOMENTUM_INTEGRATION_LMAX = -3.482155

    print("Using manual HKL integration bounds:")
    print(f"  H: [{MOMENTUM_INTEGRATION_HMIN:.6f}, {MOMENTUM_INTEGRATION_HMAX:.6f}]")
    print(f"  K: [{MOMENTUM_INTEGRATION_KMIN:.6f}, {MOMENTUM_INTEGRATION_KMAX:.6f}]")
    print(f"  L: [{MOMENTUM_INTEGRATION_LMIN:.6f}, {MOMENTUM_INTEGRATION_LMAX:.6f}]")

### Integrate Reference Momentum ROI

Integrate the selected HKL boundary, subtract the scan-specific background level if available, and save centroid/intensity diagnostics.

In [ ]:
if RUN_REFERENCE_MOMENTUM_INTEGRATION:
    integration_bounds = {
        "hmin": MOMENTUM_INTEGRATION_HMIN,
        "hmax": MOMENTUM_INTEGRATION_HMAX,
        "kmin": MOMENTUM_INTEGRATION_KMIN,
        "kmax": MOMENTUM_INTEGRATION_KMAX,
        "lmin": MOMENTUM_INTEGRATION_LMIN,
        "lmax": MOMENTUM_INTEGRATION_LMAX,
    }
    missing = [name for name, value in integration_bounds.items() if value is None]
    if missing:
        raise ValueError(
            "Set integration bounds or run the line-scan/background crossing cell first: "
            + ", ".join(missing)
        )

    background_level = None
    diagnostic_background_levels = None
    if "background_summary" in globals():
        background_level = background_summary["mean_counts_per_occupied_momentum_voxel"]
        diagnostic_background_levels = momentum_line_scan_background_levels(reference_line_scans, background_level)

    reference_integration_path = (
        repo_root
        / reference_momentum_settings["output_directory"]
        / "Transforms"
        / f"{REFERENCE_MOMENTUM_PEAK_ID}_momentum_integration.json"
    )
    reference_integration_fig_path = (
        repo_root
        / config["project"]["output_root"]
        / "Graphs"
        / REFERENCE_MOMENTUM_SCAN_ID
        / REFERENCE_MOMENTUM_PEAK_ID
        / f"{REFERENCE_MOMENTUM_PEAK_ID}_momentum_integration_diagnostic.png"
    )

    reference_integration = integrate_momentum_roi(
        transform_data=transform_data,
        Hrange=Hrange,
        Krange=Krange,
        Lrange=Lrange,
        bounds=integration_bounds,
        background_level=background_level,
        save_path=reference_integration_path,
    )
    fig, axes = plot_momentum_line_scans(
        reference_line_scans,
        save_path=reference_integration_fig_path,
        title="Reference Momentum Integration",
        background_y=diagnostic_background_levels,
        background_label="Scaled background",
        integration_bounds=reference_integration["bounds"],
    )

    print(f"Reference raw intensity: {reference_integration['raw_intensity']:.6g}")
    print(f"Reference net intensity: {reference_integration['net_intensity']:.6g}")
    print(f"Reference positive net intensity: {reference_integration['positive_net_intensity']:.6g}")
    print(f"Reference centroid HKL: {reference_integration['centroid_hkl']}")
    print(f"Reference max HKL: {reference_integration['max_hkl']}")
    print(f"Occupied voxels: {reference_integration['occupied_voxel_count']} / {reference_integration['voxel_count']}")
    print(f"Saved reference integration: {reference_integration_path}")
    print(f"Saved integration diagnostic: {reference_integration_fig_path}")
else:
    print("Skipped reference momentum integration. Set RUN_REFERENCE_MOMENTUM_INTEGRATION = True after choosing bounds.")


## Stage 7: Apply Analysis Across All Scans

Load saved peak-window and momentum-analysis definitions, then process selected CDW scans. Parallel execution will be across scans; peaks within one scan remain sequential to avoid concurrent access to the same NeXus/HDF5 file.


### CDW Strain-Series Automation

After choosing one detector-space CDW ROI and one detector-space background ROI, process the same CDW across strain scans. Each scan is transformed, binned, background-estimated, and integrated independently.


In [ ]:
RUN_CDW_STRAIN_SERIES = True

# Reuse the same CDW ID selected above, or set a different one here.
CDW_SERIES_ID = WINDOW_CDW_ID

# Automatically include every configured scan whose scan_id ends with _CDW1/_CDW2/_CDW3.
# This makes newly added scans available after updating the YAML, without editing this list.
CDW_SERIES_GROUP_IDS = [
    group_id
    for group_id, scans in sorted(
        scan_groups.items(),
        key=lambda item: (float(item[1][0].get("strain", 0.0)), item[0]),
    )
    if scans[0].get("role") == "strain" and group_id.endswith(f"_{CDW_SERIES_ID}")
]

saved_cdw_windows = cdw_series_inputs_from_saved_windows(repo_root, CDW_SERIES_ID)
CDW_SERIES_SIGNAL_CENTER = saved_cdw_windows["signal_center"]
CDW_SERIES_SIGNAL_BOUNDS = saved_cdw_windows["signal_bounds"]
CDW_SERIES_BACKGROUND_CENTER = saved_cdw_windows["background_center"]
CDW_SERIES_BACKGROUND_BOUNDS = saved_cdw_windows["background_bounds"]

# Integration bounds usually come from the reference background-intersection defaults.
CDW_SERIES_INTEGRATION_BOUNDS = {
    "hmin": MOMENTUM_INTEGRATION_HMIN,
    "hmax": MOMENTUM_INTEGRATION_HMAX,
    "kmin": MOMENTUM_INTEGRATION_KMIN,
    "kmax": MOMENTUM_INTEGRATION_KMAX,
    "lmin": MOMENTUM_INTEGRATION_LMIN,
    "lmax": MOMENTUM_INTEGRATION_LMAX,
}

print(f"Processing {CDW_SERIES_ID} groups:")
for group_id in CDW_SERIES_GROUP_IDS:
    print(f"  {group_id}")
print(f"Signal window: {saved_cdw_windows['signal_window']['path']}")
print(f"Background window: {saved_cdw_windows['background_window']['path']}")

# If False, use CDW_SERIES_INTEGRATION_BOUNDS for every strain scan.
# If True, choose scan-specific HKL bounds from each scan's own background-crossing line scans.
CDW_SERIES_AUTO_INTEGRATION_BOUNDS = False
# Keep False to reuse saved transforms/backgrounds; turn on only when deliberately recalculating.
CDW_SERIES_OVERWRITE_EXISTING = True
CDW_SERIES_BACKGROUND_STAT = "mean"

In [ ]:
if RUN_CDW_STRAIN_SERIES:
    missing_bounds = [key for key, value in CDW_SERIES_INTEGRATION_BOUNDS.items() if value is None]
    if missing_bounds:
        raise ValueError("Set CDW_SERIES_INTEGRATION_BOUNDS before running: " + ", ".join(missing_bounds))

    cdw_series_result = process_cdw_strain_series(
        cdw_id=CDW_SERIES_ID,
        scan_groups=scan_groups,
        group_ids=CDW_SERIES_GROUP_IDS,
        repo_root=repo_root,
        config=config,
        nxprocess_lab=nxprocess_lab,
        signal_center=CDW_SERIES_SIGNAL_CENTER,
        signal_bounds=CDW_SERIES_SIGNAL_BOUNDS,
        background_center=CDW_SERIES_BACKGROUND_CENTER,
        background_bounds=CDW_SERIES_BACKGROUND_BOUNDS,
        deltah=MOMENTUM_DELTAH,
        deltak=MOMENTUM_DELTAK,
        deltal=MOMENTUM_DELTAL,
        integration_bounds=CDW_SERIES_INTEGRATION_BOUNDS,
        background_stat=CDW_SERIES_BACKGROUND_STAT,
        overwrite_existing=CDW_SERIES_OVERWRITE_EXISTING,
        auto_integration_bounds=CDW_SERIES_AUTO_INTEGRATION_BOUNDS,
        hscan_min=MOMENTUM_HSCAN_MIN,
        hscan_max=MOMENTUM_HSCAN_MAX,
        kscan_min=MOMENTUM_KSCAN_MIN,
        kscan_max=MOMENTUM_KSCAN_MAX,
        lscan_min=MOMENTUM_LSCAN_MIN,
        lscan_max=MOMENTUM_LSCAN_MAX,
        hscan_step=MOMENTUM_SCAN_HSTEP,
        kscan_step=MOMENTUM_SCAN_KSTEP,
        lscan_step=MOMENTUM_SCAN_LSTEP,
    )
    print(f"Saved CDW strain summary: {cdw_series_result['summary_csv']}")
    print(f"Saved CDW strain plot: {cdw_series_result['plot_path']}")
else:
    print("Skipped CDW strain-series automation. Set RUN_CDW_STRAIN_SERIES = True after choosing ROIs and integration bounds.")


In [ ]:
from xrd_strain_pipeline.helper import plot_cdw_strain_intensity

positive_plot_path = (
    repo_root
    / config["project"]["output_root"]
    / "CDW_Analysis"
    / CDW_SERIES_ID
    / f"{CDW_SERIES_ID}_positive_net_intensity_vs_strain.png"
)

fig, ax = plot_cdw_strain_intensity(
    cdw_series_result["rows"],
    save_path=positive_plot_path,
    intensity_key="positive_net_intensity",
    error_key="poisson_sigma_raw",
)

print(f"Saved CDW positive-net strain plot: {positive_plot_path}")

In [ ]:
RUN_CDW_SERIES_DIAGNOSTICS = False

# Optional scan-specific PNG override. By default this uses the first part of DATA_directory, e.g.
# demo_strain050/CDW1 -> Max_of_All/demo_strain050.png.
CDW_SERIES_OVERLAY_FILES = {}

# Plot only the detector signal window on each strain Max-of-All PNG.
CDW_SERIES_PLOT_DETECTOR_WINDOWS = True

# Plot per-strain H/K/L line scans with integration bounds.
CDW_SERIES_PLOT_INTEGRATION_DIAGNOSTICS = True

# Keep False unless you specifically want the background line in the diagnostic plots.
CDW_SERIES_SHOW_BACKGROUND_IN_DIAGNOSTICS = False


In [ ]:
if RUN_CDW_SERIES_DIAGNOSTICS:
    if "cdw_series_result" not in globals():
        raise ValueError("Run the CDW strain-series cell first so cdw_series_result exists.")
    print("Calculated CDW series background levels:")
    for row in cdw_series_result["rows"]:
        group_id = row["scan_group_id"]
        result = cdw_series_result["group_results"][group_id]
        summaries = result.get("background_summaries", [])
        integration = result["integration"]
        print(
            f"  {group_id}: "
            f"background/occupied voxel={integration['background_per_occupied_voxel']:.6g}, "
            f"occupied_voxels={integration['occupied_voxel_count']}, "
            f"background_total={integration['background_total']:.6g}, "
            f"raw={integration['raw_intensity']:.6g}, "
            f"net={integration['net_intensity']:.6g}"
        )
        for summary in summaries:
            print(
                f"    background patch: "
                f"mean/occupied voxel={summary.get('mean_counts_per_occupied_momentum_voxel', float('nan')):.6g}, "
                f"median/occupied voxel={summary.get('median_counts_per_occupied_momentum_voxel', float('nan')):.6g}, "
                f"std={summary.get('std_counts_per_occupied_momentum_voxel', float('nan')):.6g}, "
                f"mean/detector pixel={summary.get('mean_counts_per_detector_pixel', float('nan')):.6g}"
            )


    if CDW_SERIES_PLOT_DETECTOR_WINDOWS:
        cdw_series_detector_window_paths = plot_cdw_series_detector_windows(
            cdw_series_result,
            scan_groups=scan_groups,
            repo_root=repo_root,
            config=config,
            signal_bounds=CDW_SERIES_SIGNAL_BOUNDS,
            signal_center=CDW_SERIES_SIGNAL_CENTER,
            background_bounds=None,
            background_center=None,
            overlay_files=CDW_SERIES_OVERLAY_FILES,
            crop_margin=40,
        )

    if CDW_SERIES_PLOT_INTEGRATION_DIAGNOSTICS:
        cdw_series_integration_diagnostic_paths = plot_cdw_series_integration_diagnostics(
            cdw_series_result,
            repo_root=repo_root,
            config=config,
            hmin=MOMENTUM_HSCAN_MIN,
            hmax=MOMENTUM_HSCAN_MAX,
            kmin=MOMENTUM_KSCAN_MIN,
            kmax=MOMENTUM_KSCAN_MAX,
            lmin=MOMENTUM_LSCAN_MIN,
            lmax=MOMENTUM_LSCAN_MAX,
            hstep=MOMENTUM_SCAN_HSTEP,
            kstep=MOMENTUM_SCAN_KSTEP,
            lstep=MOMENTUM_SCAN_LSTEP,
            show_background=CDW_SERIES_SHOW_BACKGROUND_IN_DIAGNOSTICS,
        )
else:
    print("Skipped CDW strain-series diagnostics.")


## Stage 8: Final Plotting

Create final intensity-versus-voltage figures from the saved CDW strain-series summaries. Actual strain conversion and normalized actual-strain plots can be added here after the strain-gauge conversion is finalized.


In [ ]:
RUN_FINAL_VOLTAGE_PLOTS = False

# Add additional CDW series after their Stage 7 summaries have been generated.
FINAL_CDW_SERIES_IDS = ["CDW1"]

# The current summary column named "strain" is the scan-control value used in the YAML.
FINAL_INTENSITY_KEY = "net_intensity"
FINAL_ERROR_KEY = "poisson_sigma_raw"
FINAL_REFERENCE_VOLTAGE = 0.0

# Kept for later: control-value/resistance -> actual strain conversion and actual-strain plots.
RUN_FINAL_ACTUAL_STRAIN_PLOTS = False
RUN_FINAL_NORMALIZED_ACTUAL_STRAIN_PLOTS = False


In [ ]:
if RUN_FINAL_VOLTAGE_PLOTS:
    final_plot_dir = repo_root / config["project"]["output_root"] / "Final_Plots"
    final_plot_dir.mkdir(parents=True, exist_ok=True)
    final_series_rows = {}

    for cdw_id in FINAL_CDW_SERIES_IDS:
        summary_path = (
            repo_root
            / config["project"]["output_root"]
            / "CDW_Analysis"
            / cdw_id
            / f"{cdw_id}_strain_intensity_summary.csv"
        )
        if not summary_path.exists():
            print(f"Skipped {cdw_id}; missing summary: {summary_path}")
            continue

        rows = load_cdw_series_summary(summary_path)
        final_series_rows[cdw_id] = rows

        _, _ = plot_final_cdw_voltage_series(
            rows,
            save_path=final_plot_dir / f"{cdw_id}_net_intensity_vs_voltage.png",
            cdw_label=cdw_id,
            intensity_key=FINAL_INTENSITY_KEY,
            error_key=FINAL_ERROR_KEY,
            normalize=False,
            reference_voltage=FINAL_REFERENCE_VOLTAGE,
            xlabel="Voltage / strain setting",
        )
        print(f"Saved {cdw_id} net intensity vs voltage plot")

    if final_series_rows:
        _, _ = plot_final_cdw_voltage_overlay(
            final_series_rows,
            save_path=final_plot_dir / "all_cdw_net_intensity_vs_voltage.png",
            intensity_key=FINAL_INTENSITY_KEY,
            error_key=FINAL_ERROR_KEY,
            normalize=False,
            reference_voltage=FINAL_REFERENCE_VOLTAGE,
            xlabel="Voltage / strain setting",
        )
        print(f"Saved combined net intensity vs voltage plot: {final_plot_dir / 'all_cdw_net_intensity_vs_voltage.png'}")

        _, _ = plot_final_cdw_voltage_overlay(
            final_series_rows,
            save_path=final_plot_dir / "all_cdw_normalized_net_intensity_vs_voltage.png",
            intensity_key=FINAL_INTENSITY_KEY,
            error_key=FINAL_ERROR_KEY,
            normalize=True,
            reference_voltage=FINAL_REFERENCE_VOLTAGE,
            xlabel="Voltage / strain setting",
        )
        print(f"Saved combined normalized net intensity vs voltage plot: {final_plot_dir / 'all_cdw_normalized_net_intensity_vs_voltage.png'}")
else:
    print("Skipped final voltage plots.")

if RUN_FINAL_ACTUAL_STRAIN_PLOTS or RUN_FINAL_NORMALIZED_ACTUAL_STRAIN_PLOTS:
    print("Actual-strain plotting is reserved for the strain-gauge conversion stage.")


## Session Snapshot

Save the current notebook namespace so later sessions can restore analysis variables without rerunning every expensive transform. The saver skips modules, functions, and unpickleable objects.


In [ ]:
from pathlib import Path

RUN_SAVE_NOTEBOOK_SESSION = False
RUN_LOAD_NOTEBOOK_SESSION = True

try:
    _snapshot_repo_root = repo_root
except NameError:
    _snapshot_repo_root = Path.cwd()
    if _snapshot_repo_root.name != "XRD_Strain_Analysis_Pipeline":
        candidate = _snapshot_repo_root / "XRD_Strain_Analysis_Pipeline"
        if candidate.exists():
            _snapshot_repo_root = candidate

try:
    _snapshot_output_root = config["project"]["output_root"]
except NameError:
    _snapshot_output_root = "Processed_Data"

SESSION_SNAPSHOT_PATH = (
    _snapshot_repo_root
    / _snapshot_output_root
    / "Session_Snapshots"
    / "main_pipeline_latest_session.pkl"
)


In [ ]:
import pickle
import types
from pathlib import Path

def _is_snapshot_candidate(name, value):
    if name.startswith("_"):
        return False
    snapshot_bookkeeping = {
        "In", "Out", "exit", "quit", "get_ipython",
        "saved_session", "skipped_session", "loaded_session",
        "_snapshot_repo_root", "_snapshot_output_root",
    }
    if name in snapshot_bookkeeping:
        return False
    if isinstance(value, types.ModuleType):
        return False
    if callable(value):
        return False
    value_module = type(value).__module__
    if value_module.startswith("nexusformat"):
        return False
    if value_module.startswith("h5py"):
        return False
    if type(value).__name__ in {"NXroot", "NXentry", "NXfield", "NXgroup", "NXdata", "NXFile", "AttrDict"}:
        return False
    return True

def save_notebook_session(path, namespace):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    saved = {}
    skipped = {}
    records = []
    for name, value in namespace.items():
        if not _is_snapshot_candidate(name, value):
            continue
        try:
            record = pickle.dumps((name, value), protocol=pickle.HIGHEST_PROTOCOL)
        except Exception as exc:
            skipped[name] = type(exc).__name__
            continue
        saved[name] = value
        records.append(record)
    tmp_path = path.with_suffix(path.suffix + ".tmp")
    with tmp_path.open("wb") as stream:
        pickle.dump({"format": "notebook_session_records_v2", "count": len(records)}, stream, protocol=pickle.HIGHEST_PROTOCOL)
        for record in records:
            stream.write(len(record).to_bytes(8, "little"))
            stream.write(record)
    tmp_path.replace(path)
    return saved, skipped

def load_notebook_session(path, namespace):
    path = Path(path)
    loaded = {}
    skipped = {}
    with path.open("rb") as stream:
        try:
            header = pickle.load(stream)
        except Exception as exc:
            raise RuntimeError(
                "Could not load this session snapshot. It was probably saved by the older "
                "all-at-once snapshot code and contains an object that cannot be restored. "
                "Rerun the notebook once, save a fresh snapshot with the updated snapshot cell, "
                "and future loads will be variable-by-variable."
            ) from exc
        if isinstance(header, dict) and header.get("format") == "notebook_session_records_v2":
            for _ in range(header["count"]):
                size_bytes = stream.read(8)
                if not size_bytes:
                    break
                record = stream.read(int.from_bytes(size_bytes, "little"))
                try:
                    name, value = pickle.loads(record)
                except Exception as exc:
                    skipped[f"record_{len(loaded) + len(skipped)}"] = type(exc).__name__
                    continue
                loaded[name] = value
        elif isinstance(header, dict):
            loaded = header
        else:
            raise ValueError(f"Unrecognized session snapshot format: {type(header).__name__}")
    namespace.update(loaded)
    return loaded, skipped

if RUN_LOAD_NOTEBOOK_SESSION:
    loaded_session, skipped_load_session = load_notebook_session(SESSION_SNAPSHOT_PATH, globals())
    print(f"Loaded {len(loaded_session)} variables from: {SESSION_SNAPSHOT_PATH}")
    if skipped_load_session:
        print(f"Skipped {len(skipped_load_session)} variables while loading.")

if RUN_SAVE_NOTEBOOK_SESSION:
    saved_session, skipped_session = save_notebook_session(SESSION_SNAPSHOT_PATH, globals())
    print(f"Saved {len(saved_session)} variables to: {SESSION_SNAPSHOT_PATH}")
    if skipped_session:
        print(f"Skipped {len(skipped_session)} unpickleable variables.")
        print("First skipped variables:", sorted(skipped_session)[:20])
else:
    print("Skipped notebook session save.")
